In [2]:
# 1. Import SparkSession
from pyspark.sql import SparkSession


In [3]:
# 2. Create SparkSession
spark = SparkSession.builder.appName("PySpark_Interview_Demo").getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/05 00:10:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/05 00:10:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
spark

In [5]:
# 3. Load Silver data (already cleaned)
parts = spark.read.parquet("silver/parts/")
prod = spark.read.parquet("silver/production/")
qual = spark.read.parquet("silver/quality/")
maint = spark.read.parquet("silver/maintenance/")


In [6]:
# 4. Show schema (understand structure)
parts.printSchema()


root
 |-- part_id: string (nullable = true)
 |-- material: string (nullable = true)
 |-- volume_mm3: double (nullable = true)
 |-- surface_area_mm2: double (nullable = true)
 |-- tolerance_um: double (nullable = true)
 |-- complexity_idx: double (nullable = true)
 |-- rev: integer (nullable = true)
 |-- created_at: timestamp (nullable = true)



In [7]:
parts.show(5)

+-------+--------+----------+----------------+------------+--------------+---+-------------------+
|part_id|material|volume_mm3|surface_area_mm2|tolerance_um|complexity_idx|rev|         created_at|
+-------+--------+----------+----------------+------------+--------------+---+-------------------+
|P000001|Aluminum|    6705.0|           373.0|        67.0|         0.535|  1|2025-02-05 00:00:00|
|P000002|     ABS|   16180.0|           716.0|        80.0|         0.372|  1|2025-02-22 00:00:00|
|P000003|   Steel|    9677.0|           462.0|        20.0|         0.365|  1|2025-01-05 00:00:00|
|P000004|     ABS|    3755.0|           263.0|        88.0|         0.508|  1|2025-02-12 00:00:00|
|P000005|Titanium|   36398.0|          1256.0|        37.0|         0.152|  1|2025-01-13 00:00:00|
+-------+--------+----------+----------------+------------+--------------+---+-------------------+
only showing top 5 rows


In [9]:
print("Total parts:", parts.count())
print("Total production records:", prod.count())
print("Total quality records:", qual.count())
print("Total maintenance records:", maint.count())


Total parts: 20000
Total production records: 169721
Total quality records: 28363
Total maintenance records: 52


In [12]:
# 7. Select specific columns
parts.select("part_id", "material", "tolerance_um").show(5)


+-------+--------+------------+
|part_id|material|tolerance_um|
+-------+--------+------------+
|P000001|Aluminum|        67.0|
|P000002|     ABS|        80.0|
|P000003|   Steel|        20.0|
|P000004|     ABS|        88.0|
|P000005|Titanium|        37.0|
+-------+--------+------------+
only showing top 5 rows


In [15]:
parts.filter(parts.material == "Steel").show(15)

+-------+--------+----------+----------------+------------+--------------+---+-------------------+
|part_id|material|volume_mm3|surface_area_mm2|tolerance_um|complexity_idx|rev|         created_at|
+-------+--------+----------+----------------+------------+--------------+---+-------------------+
|P000003|   Steel|    9677.0|           462.0|        20.0|         0.365|  1|2025-01-05 00:00:00|
|P000006|   Steel|   14731.0|           632.0|        52.0|         0.272|  1|2025-01-01 00:00:00|
|P000007|   Steel|   16903.0|           636.0|        38.0|         0.199|  1|2025-02-16 00:00:00|
|P000010|   Steel|    8541.0|           466.0|        29.0|         0.845|  1|2025-02-22 00:00:00|
|P000013|   Steel|   24603.0|           873.0|        41.0|         0.481|  1|2025-01-11 00:00:00|
|P000015|   Steel|   20601.0|           852.0|        28.0|         0.758|  1|2025-02-28 00:00:00|
|P000018|   Steel|    8702.0|           459.0|        30.0|         0.502|  1|2025-02-25 00:00:00|
|P000024| 

In [17]:
parts.groupBy("material").avg("tolerance_um").show()
parts.groupBy("material").count().show()
parts.groupBy("material").max("tolerance_um").show()
parts.groupBy("material").min("tolerance_um").show()
parts.groupBy("material").sum("tolerance_um").show()
parts.groupBy("material").mean("tolerance_um").show()

+--------+------------------+
|material| avg(tolerance_um)|
+--------+------------------+
|     ABS| 75.21518691588786|
|   Steel| 35.69557229469043|
|Aluminum| 55.38013698630137|
| UNKNOWN| 51.34164588528678|
|Titanium|31.156234096692113|
|   Brass| 45.26237833262802|
+--------+------------------+

+--------+-----+
|material|count|
+--------+-----+
|     ABS| 4280|
|   Steel| 5443|
|Aluminum| 5548|
| UNKNOWN|  401|
|Titanium| 1965|
|   Brass| 2363|
+--------+-----+

+--------+-----------------+
|material|max(tolerance_um)|
+--------+-----------------+
|     ABS|            114.0|
|   Steel|             84.0|
|Aluminum|            101.0|
| UNKNOWN|            104.0|
|Titanium|             75.0|
|   Brass|             84.0|
+--------+-----------------+

+--------+-----------------+
|material|min(tolerance_um)|
+--------+-----------------+
|     ABS|             33.0|
|   Steel|             15.0|
|Aluminum|             15.0|
| UNKNOWN|             15.0|
|Titanium|             15.0|
|   B

In [18]:
# 10. Scrap rate per machine
from pyspark.sql.functions import avg, col
prod.groupBy("machine_id").agg(avg(col("scrap_flag").cast("int")).alias("scrap_rate")).show(10)


+----------+-------------------+
|machine_id|         scrap_rate|
+----------+-------------------+
|      M002|0.14759928867812686|
|      M011|0.14432748538011697|
|      M003|0.14885406922357344|
|      M008|  0.136903348373308|
|      M015|0.14516129032258066|
|      M004|0.14401236770127246|
|      M006|0.14241486068111456|
|      M019| 0.1447736817545497|
|      M014|0.14503905794566865|
|      M001|0.14656793606017865|
+----------+-------------------+
only showing top 10 rows


In [21]:
# 11. Count number of inspections per part
qual.groupBy("part_id").count().show(15)


+-------+-----+
|part_id|count|
+-------+-----+
|P000083|    1|
|P000235|    1|
|P000312|    1|
|P000604|    2|
|P001111|    2|
|P001195|    1|
|P001267|    1|
|P001325|    2|
|P002678|    2|
|P003061|    2|
|P003292|    2|
|P003446|    2|
|P003908|    2|
|P004068|    1|
|P004408|    2|
+-------+-----+
only showing top 15 rows


In [22]:
# 12. Average downtime per machine
maint.groupBy("machine_id").avg("downtime_min").show(5)


+----------+------------------+
|machine_id| avg(downtime_min)|
+----------+------------------+
|      M002|109.64000000000001|
|      M011|             174.3|
|      M003|            253.75|
|      M008|            47.675|
|      M015|              36.7|
+----------+------------------+
only showing top 5 rows


In [23]:
# 13. Join parts with production
parts_prod = prod.join(parts, "part_id", "inner")
parts_prod.select("part_id", "material", "cycle_time_sec").show(5)


+-------+--------+--------------+
|part_id|material|cycle_time_sec|
+-------+--------+--------------+
|P000001|Aluminum|         20.29|
|P000001|Aluminum|          21.8|
|P000001|Aluminum|         19.05|
|P000001|Aluminum|         19.31|
|P000001|Aluminum|          24.3|
+-------+--------+--------------+
only showing top 5 rows


In [24]:
parts.show(5)
prod.show(5)

+-------+--------+----------+----------------+------------+--------------+---+-------------------+
|part_id|material|volume_mm3|surface_area_mm2|tolerance_um|complexity_idx|rev|         created_at|
+-------+--------+----------+----------------+------------+--------------+---+-------------------+
|P000001|Aluminum|    6705.0|           373.0|        67.0|         0.535|  1|2025-02-05 00:00:00|
|P000002|     ABS|   16180.0|           716.0|        80.0|         0.372|  1|2025-02-22 00:00:00|
|P000003|   Steel|    9677.0|           462.0|        20.0|         0.365|  1|2025-01-05 00:00:00|
|P000004|     ABS|    3755.0|           263.0|        88.0|         0.508|  1|2025-02-12 00:00:00|
|P000005|Titanium|   36398.0|          1256.0|        37.0|         0.152|  1|2025-01-13 00:00:00|
+-------+--------+----------+----------------+------------+--------------+---+-------------------+
only showing top 5 rows
+----------+-------------------+-------+--------------+----------+------------+------

In [26]:
# 14. Join with quality to see failures
parts_all = parts.join(qual, "part_id", "left").join(prod, "part_id", "left")
parts_all.select("part_id", "material", "status", "scrap_flag").show(5)
parts_all.groupBy("material").agg(avg(col("scrap_flag").cast("int")).alias("scrap_rate")).show()

+-------+--------+------+----------+
|part_id|material|status|scrap_flag|
+-------+--------+------+----------+
|P000001|Aluminum|  PASS|     false|
|P000001|Aluminum|  PASS|     false|
|P000001|Aluminum|  PASS|     false|
|P000001|Aluminum|  PASS|     false|
|P000001|Aluminum|  PASS|     false|
+-------+--------+------+----------+
only showing top 5 rows
+--------+-------------------+
|material|         scrap_rate|
+--------+-------------------+
|     ABS| 0.0820912504266697|
|   Steel|0.19820023514515692|
|Aluminum| 0.1146551977218683|
| UNKNOWN|0.14418229481750353|
|Titanium|0.21506723461120855|
|   Brass|0.14570451483643648|
+--------+-------------------+



In [27]:
# 15. Add computed column: cycle_time in minutes
prod = prod.withColumn("cycle_time_min", col("cycle_time_sec")/60)
prod.select("part_id", "cycle_time_sec", "cycle_time_min").show(5)


+-------+--------------+-------------------+
|part_id|cycle_time_sec|     cycle_time_min|
+-------+--------------+-------------------+
|P000001|         20.29|0.33816666666666667|
|P000001|          21.8|0.36333333333333334|
|P000001|         19.05|             0.3175|
|P000001|         19.31| 0.3218333333333333|
|P000001|          24.3|              0.405|
+-------+--------------+-------------------+
only showing top 5 rows


In [29]:
# 16. Handle null values: fill missing operator_id
prod_filled = prod.fillna({"operator_id": "UNKNOWN"})
prod_filled.select("part_id", "operator_id").show(50)


+-------+-----------+
|part_id|operator_id|
+-------+-----------+
|P000001|       O001|
|P000001|       O002|
|P000001|       O001|
|P000001|       O001|
|P000001|       O007|
|P000001|       O002|
|P000001|       O001|
|P000001|       O004|
|P000001|       O001|
|P000001|       O007|
|P000002|       O009|
|P000002|       O007|
|P000002|       O002|
|P000002|       O009|
|P000002|       O010|
|P000003|       O009|
|P000003|       O001|
|P000003|       O008|
|P000003|       O006|
|P000003|       O008|
|P000003|       O003|
|P000003|       O005|
|P000003|       O006|
|P000004|       O002|
|P000004|       O007|
|P000004|       O003|
|P000004|       O001|
|P000004|       O010|
|P000004|       O006|
|P000004|       O010|
|P000004|       O005|
|P000004|       O001|
|P000004|       O008|
|P000004|       O006|
|P000004|       O006|
|P000005|       O008|
|P000005|       O005|
|P000005|       O007|
|P000005|       O002|
|P000005|       O009|
|P000005|       O007|
|P000005|       O007|
|P000006| 

In [30]:
# 17. Drop rows with missing energy_kwh
prod_nonull = prod.dropna(subset=["energy_kwh"])
print("Rows after dropping null energy:", prod_nonull.count())


Rows after dropping null energy: 169721


In [31]:
# 18. Use window to rank parts by avg cycle time
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

w = Window.orderBy(col("avg_cycle_time_sec").desc())
avg_cycle = prod.groupBy("part_id").avg("cycle_time_sec").withColumnRenamed("avg(cycle_time_sec)", "avg_cycle_time_sec")
ranked = avg_cycle.withColumn("rank", rank().over(w))
ranked.show(10)


+-------+------------------+----+
|part_id|avg_cycle_time_sec|rank|
+-------+------------------+----+
|P008818|            74.505|   1|
|P002049| 74.36285714285714|   2|
|P007465|           73.8225|   3|
|P010593| 73.52000000000001|   4|
|P008112| 72.38416666666667|   5|
|P006005| 72.31666666666666|   6|
|P003852| 71.77000000000001|   7|
|P010460| 71.55499999999999|   8|
|P001786| 71.35875000000001|   9|
|P000211|            71.292|  10|
+-------+------------------+----+
only showing top 10 rows


25/09/05 00:29:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/05 00:29:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/05 00:29:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/05 00:29:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/05 00:29:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/05 00:29:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [32]:
# 19. Calculate correlation between complexity and cycle time
from pyspark.sql.functions import corr
parts_prod_corr = parts_prod.corr("complexity_idx", "cycle_time_sec")
print("Correlation complexity vs cycle time:", parts_prod_corr)


Correlation complexity vs cycle time: 0.31757974364352465


In [34]:
# 20. Assemble features for ML
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["volume_mm3", "surface_area_mm2", "complexity_idx", "tolerance_um"],
    outputCol="features"
)
ml_df = assembler.transform(parts)
ml_df.select("part_id", "features").show(5, truncate=False)


+-------+---------------------------+
|part_id|features                   |
+-------+---------------------------+
|P000001|[6705.0,373.0,0.535,67.0]  |
|P000002|[16180.0,716.0,0.372,80.0] |
|P000003|[9677.0,462.0,0.365,20.0]  |
|P000004|[3755.0,263.0,0.508,88.0]  |
|P000005|[36398.0,1256.0,0.152,37.0]|
+-------+---------------------------+
only showing top 5 rows


In [35]:
# 21. Train simple regression model on synthetic cost
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import expr

# Add synthetic label
ml_df = ml_df.withColumn("cost", expr("0.001*volume_mm3 + 0.1*tolerance_um + 5"))

lr = LinearRegression(featuresCol="features", labelCol="cost")
model = lr.fit(ml_df)

print("Coefficients:", model.coefficients)
print("Intercept:", model.intercept)


25/09/05 00:31:18 WARN Instrumentation: [0c2344ee] regParam is zero, which might cause numerical instability and overfitting.
25/09/05 00:31:18 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/09/05 00:31:18 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Coefficients: [0.0010000000000007561,-2.909994042550127e-14,3.71653501547266e-13,0.09999999999999738]
Intercept: 5.000000000006149


In [36]:
# 22. Make predictions
pred = model.transform(ml_df)
pred.select("part_id", "cost", "prediction").show(5)


+-------+------------------+------------------+
|part_id|              cost|        prediction|
+-------+------------------+------------------+
|P000001|            18.405| 18.40500000000039|
|P000002|             29.18|29.179999999997477|
|P000003|            16.677|16.677000000000103|
|P000004|            17.555|17.555000000001293|
|P000005|45.098000000000006| 45.09799999999708|
+-------+------------------+------------------+
only showing top 5 rows


In [37]:
# 23. Save model
model.write().overwrite().save("models/demo_linear_model")


In [38]:
# 24. Save transformed features
ml_df.write.mode("overwrite").parquet("gold/demo_features/")


25/09/05 01:00:51 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 742799 ms exceeds timeout 120000 ms
25/09/05 01:00:51 WARN SparkContext: Killing executors is not supported by current scheduler.
25/09/05 01:00:51 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$